In [ ]:

#单个股票价值的计算
import pandas as pd

'''VaR'''

'''to get the value of 100 shares of IBM stock'''
import pandas as pd
imoprt yfinance as yf
ticker = "IBM"
df = yf.download(ticker)
df = pd.read_pickle("ibm.pkl")
nStocks = 100
wealth = nStocks * df["Close"].iloc[-1]
wealth = round(wealth, 2)
print(f"the value of {nStocks} shares of IBM is {wealth}")






In [ ]:
#单日var的计算
#to calculate the var next day for 1000 share of ibm
from scipy.stats import norm
ticker='IBM'
n_shares=1000
confidence_level=0.99
z=norm.ppf(1-confidence_level)

ret=df['Adj Close'].pct_change().dropna()
position=round(n_shares*df['Adj Close'][-1],2)
stdev=ret.std()
mean=ret.mean()
var=round(position*(mean-z*stdev),2)

#simplied
var_s=round(position*(z*stdev),2)

lastday=df.index[-1].date()
print(f"hoding={position}, var={var}")
print(f'simplied var={var_s} tomorrow')
print(f'number of shares={n_shares}, today={lastday}')


In [ ]:
#多日var计算，以10天为一组
#how to calculate 10-days return
import numpy as np
from math import log,exp
ticker='WMT'
ndays=10

df=yf.download(ticker)
df= pd.read_pickle('wmt.pkl')

df['retplus]=df['Adj Close'].pct_change().dropna()+1
ddate=[]
for i in range(0,df.shape[0]):
    ddate.append(int(i/ndays))
df['ndays']=ddate

ret2=df.retplus.groupby(df.ndays).prod()-1

#10-days var (calculate  10-days return)
from scipy.stats import norm
n_shares=50
confidence_level=0.99
n_days=10
z=norm.ppf(confidence_level)

df=pd.read_pickle('wmt2.pkl')
df[retplus1]=df['Adj Close'].pct_change().dropna()+1
ddate=[]
for i in range(0,df.shape[0]):
    ddate.append(int(i/ndays))
df['ndays']=ddate
ret2=df.retplus1.groupby(df.ndays).prod()-1
mean=ret2.mean()
stdev=ret2.std()
position=round(n_shares*df['Adj Close'][0],2)
var=round(position*(mean-z*stdev),2)
print(f'number of shares={n_shares} for wmt')
print(f'holding={position}, var={var} in {n_days} days')

In [ ]:
#正态性检验，检验真实的股票收益是否真的服从正态分布
#normality test for wmts daily return
ret=df['Adj Close'].pct_change().dropna()
print(stats.shapiro(ret))
print(stats.anderson(ret['WMT']))



In [ ]:
#var的非参数方法——排序法，不假设任何分布，直接从历史数据中找第 1% 小的日收益率，用它来衡量风险
#var for the next day (by sorting)
n_shares=500
confidence_level=0.99
df['ret']=df['Adj Close'].pct_change().dropna()
ret2=np.sort(df['ret'])
position=round(n_shares*df['Adj Close'][-1],2)
n=len(ret2)
m=int(n*(1-confidence_level))

var_sort=round(position*ret2[m],2)
print(f'var sorted={var_sort}  next day ,wealth={position}')



In [ ]:
#计算排序法下10天的var
#10-days var with sorting (calculate  10-days return)
ddate=[]
for i in range(0,df.shape[0]):
    ddate.append(int(i/ndays))
df['ndays']=ddate
ret2=df.retplus1.groupby(df.ndays).prod()-1
ret3=np.sort(ret2)
n=len(ret3)
lefttail=int(n*(1-confidence_level))
var_sort=round(position*ret3[lefttail],2)

print(f'holding={position}, var={var_sort} in {n_days} days')

In [ ]:
#calculate skewness and kurtosis 计算偏度和峰度
from numpy import mean, std, power,random
np.random.seed(12345)
n=5000000
ret=random.normal(loc=0, scale=1, size=n)
mean=ret.mean()
std=ret.std()

skew=0
kurt=0
for r in range(ret):
    skew+=power((r-mean)/std,3)
    kurt+=power((r-mean)/std,4)

skew/=(std**3)*(n-1)
kurt/=(std**4)*(n-1)
print(f'skewness={skew}, kurtosis={kurt}')


In [ ]:
#var for one day
confidence_level=0.99
n_shares=500
z=stats.norm.ppf(1-confidence_level)
ret=df['Adj Close'].pct_change().dropna()
position=round(n_shares*df['Adj Close'][-1],2)
var1=round(position*(mean-z*ret.std),2)



In [ ]:
#改进的var计算方法，基于4阶矩
#modified var based on 4 moments
s=stats.skew(ret)
k=stats.kurtosis(ret)
t=z+1/6*(z**2-1)*s+1/24*(z**3-3*z)*k-1/36*(2*z**3-5*z)*s**2
var2=round(position*(mean-t*ret.std),2)
print(f'hoding={position}, var={var1}, modified_var={var2} next day')


In [ ]:
#expected shortfall
ret2=np.sort(df['ret'])
position=round(n_shares*df['Adj Close'][-1],2)
n=ret2.shape[0]
m=int(n*(1-confidence_level))

sum=0.0
for i in np.arange(m):
    sum+=ret2[i]
ret2=sum/m
es=round(position*(ret2),2)
print(f'expected shortfall={es} ')


In [ ]:
#volatility of 2 stock portfolio 两资产组合的波动率
import numpy as np
from math import sqrt
def portfolio_volatility(ret1, ret2, w1):
    x1=w1
    x2=1-w1
    s1=ret1.std()
    s2=ret2.std()
    v=x1**2*s1**2+x2**2*s2**2+2*x1*x2*np.cov(ret1, ret2)[1][1]
    return sqrt(v)

#to randomly generate two datasets to calculate the volatility of a portfolio
from numpy import random
np.random.seed(12345)
n=50000
m1=0.02
m2=0.05
std1_0=0.2
std2_0=0.12
ret1=random.normal(loc=m1, scale=std1_0, size=n)
ret2=random.normal(loc=m2, scale=std2_0, size=n)
w1=0.4
std1=round(ret1.std(),6)
std2=round(ret2.std(),6)
p_vol=round(portfolio_volatility(ret1, ret2, w1),6)
print(f'vol of a 2-stock port ={p_vol}, w={w1}, std1={std1}, std2={std2}')

  

In [ ]:
#a realworld example to calculate the volatility of a 2-stock portfolio ibm and wmt
df=pd.read_pickle('ibm.pkl')
df=pd.DataFrame(df)
retibm=df['Adj Close'].pct_change().dropna()
retibm.columns=['retibm']

df2=pd.read_pickle('wmt.pkl')
df2=pd.DataFrame(df2)
retwmt=df2['Adj Close'].pct_change().dropna()
retwmt.columns=['retwmt']

w1=0.5
final=retibm.merge(retwmt, left_index=True, right_index=True)
std1=round(final.retibm.std(),6)
std2=round(final.retwmt.std(),6)
p_vol=round(portfolio_volatility(final.retibm, final.retwmt, w1),6)
print(f'vol of a 2-stock port ={p_vol}, w1={w1}')

w1=0.5
w2=0.3
w3=0.7
final=retibm.merge(retwmt, left_index=True, right_index=True)
std1=round(final.retibm.std(),6)
std2=round(final.retwmt.std(),6)
p_vol=round(portfolio_volatility(final.retibm, final.retwmt, w3),6)
print(f'vol of a 2-stock port ={p_vol}, w1={w2}')



In [ ]:
#var for portfolio
def vol_portfolio(ret_matrix, w):
    m=ret_matrix.shape[1]
    m2=len(w)
    if(m!=m2):
        print('the number of stocks and weights do not match')
    else:
        cov=np.cov(ret_matrix.T)
        final=0
        n=np.arange(0,m)
        for i in n:
            for j in n:
                final+=w[i]*w[j]*cov[i][j]
        std_port=sqrt(final)
        return std_port

#vol of n-stock portfolio(stimulated returns) n资产组合的波动率（模拟收益）

nreturns=500
nstocks=10
np.random.seed(12345)
ret_matrix=np.random.uniform(low=-0.12, high=0.05, size=(nreturns, nstocks))

w=np.ones(nstocks)/nstocks
vol=round(vol_portfolio(ret_matrix, w),6)
print(f'vol of a {nstocks}-stock port ={vol}, w={w}')
